In [ ]:
import pandas as pd
import os
import re
from dotenv import load_dotenv

load_dotenv()


bronze_dir = os.getenv("BRONZE")


# 1. Path Configuration
excel_path = os.getenv("BRONZE") + r"\RAW_MERGED.xlsx"


if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

# Load the file
df = pd.read_excel(excel_path)

# Clean up column names to prevent trailing/leading space KeyErrors
df.columns = df.columns.str.strip()

# Target column name variable (adjust if it's lowercase 'd' in your file)
target_col = 'Proposal Details' 

if target_col not in df.columns:
    raise KeyError(f"Could not find '{target_col}' column. Available columns are: {list(df.columns)}")

# 2. Define the Parsing Function
def extract_parivesh_details(text):
    # Fallback for empty/NaN cells
    if pd.isna(text):
        return pd.Series([None] * 5)
    
    text_str = str(text)
    
    # Using regex lookarounds to capture data between the known field labels
    clearance = re.search(r'Clearance Type:\s*(.*?)(?=\s*S/W No\.:|$)', text_str, re.IGNORECASE)
    sw_no     = re.search(r'S/W No\.\s*:\s*(.*?)(?=\s*Category:|$)', text_str, re.IGNORECASE)
    category  = re.search(r'Category\s*:\s*(.*?)(?=\s*Sector:|$)', text_str, re.IGNORECASE)
    sector    = re.search(r'Sector\s*:\s*(.*?)(?=\s*Date of Submission:|$)', text_str, re.IGNORECASE)
    date_sub  = re.search(r'Date of Submission\s*:\s*(.*?)$', text_str, re.IGNORECASE)
    
    # Extract match if found, strip trailing spaces, otherwise return None
    return pd.Series([
        clearance.group(1).strip() if clearance else None,
        sw_no.group(1).strip() if sw_no else None,
        category.group(1).strip() if category else None,
        sector.group(1).strip() if sector else None,
        date_sub.group(1).strip() if date_sub else None
    ])

# 3. Apply parsing to generate 5 new columns
print("Extracting data points from Proposal Details...")

new_cols = ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']
df[new_cols] = df[target_col].apply(extract_parivesh_details)

# 4. Save back to Excel
df.to_excel(excel_path, index=False)
print(f"Success! Extracted fields saved into columns: {new_cols}")

Extracting data points from Proposal Details...
Success! Extracted fields saved into columns: ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']


In [ ]:
#Fill Missing values for Sector

# Define your file paths
input_path = (
    os.getenv("BRONZE") + r"\RAW_MERGED.xlsx"
)
output_path = os.getenv("BRONZE") + r"\RAW_MERGED.xlsx"

# 1. Load the Excel file
df = pd.read_excel(input_path)

# Optional: Strip leading/trailing whitespaces to ensure exact matching
df["Activity Description"] = df["Activity Description"].astype(str).str.strip()

# 2. Build the lookup mapping from non-null Sector values
sector_lookup = (
    df.dropna(subset=["Sector"])
    .drop_duplicates(subset=["Activity Description"])
    .set_index("Activity Description")["Sector"]
    .to_dict()
)

# 3. Fill in missing Sector values using the Activity Description lookup
df["Sector"] = df["Sector"].fillna(df["Activity Description"].map(sector_lookup))

# 4. Save to a new Excel file
df.to_excel(output_path, index=False)

print(f"Processing complete! Saved updated file to:\n{output_path}")

Processing complete! Saved updated file to:
F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx


In [1]:
#Apply Filters and variables
import os
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Define input and output paths
input_path = os.getenv("BRONZE") + r"\RAW_MERGED.xlsx"
output_path = os.getenv("BRONZE") + r"\RAW_MERGED - IND2.xlsx"

# 1. Load Excel dataset
df = pd.read_excel(input_path)

# Ensure 'Date of Submission' is treated as datetime for accurate filtering
# (handles mixed string/datetime inputs safely)
df["Date_Parsed"] = pd.to_datetime(
    df["Date of Submission"], format="%d/%m/%Y", errors="coerce"
)

# 2. Build Filter Conditions

# Filter 1: Proposal Status
status_filter = df["Proposal Status"] == "EC Granted"

# Filter 2: Clearance Type
clearance_types = [
    "Application for EC (Category A, B1, and B2 Violation)- Form 1",
    "Application for ToR (Category A, B1, and B2 Violation)/EC (Category B2) - Form 1",
]
clearance_filter = df["Clearance Type"].isin(clearance_types)

# Filter 3: Sector
sectors = [
    "Industrial Projects - 1",
    "Industrial Projects - 2",
    "Industrial Projects - 3",
    "Non-Coal Mining",
    "Thermal Projects",
]
sector_filter = df["Sector"].isin(sectors)

# Filter 4: Date of Submission (Year 2025 or 2026)
year_filter = df["Date_Parsed"].dt.year.isin([2025, 2026])

# Filter 5: Exclude specific Activity Descriptions
excluded_activities = [
    "5(g) Distilleries",
    "5(ga) Grain based distilleries",
    "5(j) Sugar Industry",
]
activity_filter = ~df["Activity Description"].isin(excluded_activities)

# Combine all filters
combined_filter = (
    status_filter
    & clearance_filter
    & sector_filter
    & year_filter
    & activity_filter
)

# 3. Apply Filters
filtered_df = df[combined_filter].copy()

# 4. Keep specified columns only
columns_to_keep = [
    "Proposal No.",
    "Project Name",
    "Location",
    "Project Proponent",
    "Proposal Status",
    "Activity Description",
    "Clearance Type",
    "Sector",
    "Date of Submission",
]

filtered_df = filtered_df[columns_to_keep]

# 5. Save the filtered dataset to the output path
filtered_df.to_excel(output_path, index=False)

print(f"File successfully filtered and saved to: {output_path}")

File successfully filtered and saved to: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED - IND2.xlsx


In [ ]:
import os
import re
import pandas as pd
from dotenv import load_dotenv


load_dotenv()


# Define path to target Excel file
file_path = os.getenv("BRONZE") + r"\RAW_MERGED - IND1.xlsx"

if not os.path.exists(file_path):
    print(f"❌ File not found at path: {file_path}")
else:
    print(f"📂 Loading file: {file_path}")
    df = pd.read_excel(file_path)

    # 1. Strip non-printable ASCII control characters to avoid IllegalCharacterError
    illegal_xml_chars_re = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")
    df = df.map(
        lambda x: illegal_xml_chars_re.sub("", x) if isinstance(x, str) else x
    )

    # 2. Function to extract Form No from the end of string
    def extract_form_number(text):
        if pd.isna(text):
            return None
        match = re.search(
            r"Form\s*[-–]?\s*([A-Za-z0-9]+)\s*$", str(text), re.IGNORECASE
        )
        return f"Form-{match.group(1)}" if match else None

    # Identify the clearance column name dynamically (handles variations)
    clearance_col = None
    for col in df.columns:
        if "clearance" in str(col).lower():
            clearance_col = col
            break

    if clearance_col:
        print(f"🔍 Found target column: '{clearance_col}'")

        # Create new Form_No column right after the clearance type column
        col_idx = df.columns.get_loc(clearance_col) + 1
        form_series = df[clearance_col].apply(extract_form_number)
        df.insert(col_idx, "Form_No", form_series)

        # 3. Save updates back to the same Excel file
        df.to_excel(file_path, index=False)
        print(f"\n✅ File updated successfully at:\n   {file_path}")
        print("\n--- Preview of updated columns ---")
        print(df[[clearance_col, "Form_No"]].head(10))
    else:
        print(
            "❌ Could not locate a 'Clearance Type' column in the Excel file."
        )